NOTE: This is an altered copy of a notebook provided by the organizers of the Current Topics in Digital Philology course.

### **Current Topics in Digital Philology 5LN720/5LN721**

# **Lab 5: Assignment on Digital Methods in Literary Analysis**



## **1. Get started**

First, we import the Gutenberg corpus and some packages and tools that might be useful.

In [ ]:
import nltk

from nltk.corpus import gutenberg  # The Gutenberg corpus
from nltk.corpus import (
    stopwords,
)  # Stop words = a set of high frequent words in a language (e.g. “the”, “is”, “and”) that you might want to filter out
from nltk import (
    word_tokenize,
    sent_tokenize,
)  # NLTK package for tokenizing words or sentences
from nltk import WordNetLemmatizer  # NLTK package for lemmatizing words

import string
from collections import OrderedDict
import matplotlib.pyplot as plt
from collections import Counter
import re

nltk.download("gutenberg")
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("universal_tagset")
nltk.download("wordnet")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

print("Done!")

## **3. Data Selection**

In [ ]:
books = [
    "carroll-alice.txt",
    "austen-sense.txt",
]  # <------------------------------------------------  1. CHANGE TO THE BOOK(S) OF YOUR COICE!

data = []
data_sents = []
for book in books:
    data.append(gutenberg.words(book))  # data as tokens
    data_sents.append(gutenberg.sents(book))  # data as sentences, used for pos tagging

print("Example of how the data as tokens looks like:")
print((" ".join(data_sents[0][0]).replace("[ ", "").replace(" ]", "")))

train_set = [
    "austen-emma.txt",
    "austen-persuasion.txt",
    "austen-sense.txt",
    "chesterton-ball.txt",
    "chesterton-brown.txt",
    "chesterton-thursday.txt",
    "bible-kjv.txt",
    "blake-poems.txt",
    "bryant-stories.txt",
    "burgess-busterbrown.txt",
]

val_set = ["edgeworth-parents.txt", "milton-paradise.txt", "whitman-leaves.txt"]

test_set = [
    "shakespeare-caesar.txt",
    "shakespeare-hamlet.txt",
    "shakespeare-macbeth.txt",
    "melville-moby_dick.txt",
    "carroll-alice.txt",
]

# **Custom Code Intermission**

In [ ]:
import os
from copy import deepcopy
import json

import pandas as pd
import matplotlib.pyplot as plt

from featurizer import Featurizer
from classifier import Classifier


RERUN_FEATURES = False


def main():
    featurizer = Featurizer()

    full_train_set = get_features(
        featurizer,
        RERUN_FEATURES,
        feature_filename="data/train_features.csv",
        filename=train_set,
    )
    full_dev_set = get_features(
        featurizer,
        RERUN_FEATURES,
        feature_filename="data/dev_features.csv",
        filename=val_set,
    )
    full_test_set = get_features(
        featurizer,
        RERUN_FEATURES,
        feature_filename="data/test_features.csv",
        filename=test_set,
    )

    print(full_train_set["author"].value_counts())

    # exit()
    # print("Running classifier...")
    # classifier = Classifier()

    # colnames = [col for col in full_train_set.columns if col.startswith("f_")]

    # train_x = full_train_set[colnames].to_numpy()
    # train_y = full_train_set["author"].to_numpy()
    # classifier.fit(train_x, train_y)

    # print("Evaluating classifier...")
    # train_report = classifier.evaluate(train_x, train_y)

    # dev_x = full_dev_set[colnames].to_numpy()
    # dev_y = full_dev_set["author"].to_numpy()
    # dev_report = classifier.evaluate(dev_x, dev_y)

    # test_x = full_test_set[colnames].to_numpy()
    # test_y = full_test_set["author"].to_numpy()
    # test_report = classifier.evaluate(test_x, test_y)

    # report = {
    #     "train_report": train_report,
    #     "dev_report": dev_report,
    #     "test_report": test_report,
    # }
    # with open(f"classifier_results.json", "w") as outfile:
    #     json.dump(report, outfile, indent=4)

    # print("Doing ablation study")
    # perform_ablation_study(colnames, full_train_set, full_dev_set, full_test_set)
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, reverse=True
    # )
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, groups=True
    # )
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, groups=True, reverse=True
    # )


def get_features(
    featurizer: Featurizer, rerun_features: bool, feature_filename: str, filename: str
) -> pd.DataFrame:
    """Get the features for texts and return a dataframe with the text and the features

    Args:
        featurizer (Featurizer): Object that performs featurization
        rerun_features (bool): If the features should be re-calculated
        feature_filename (str): Filename for the calculated features
        filename (str): Filename containing the texts and authors

    Returns:
        pd.Dataframe: All texts, their features and their author
    """
    # data_set = pd.read_csv(filename, index_col=0)
    # data_set.index.names = ["index"]

    if rerun_features or not os.path.exists(feature_filename):
        print("Calculating features...")
        features = featurizer.featurize(filename)
        features.to_csv(feature_filename)
        summary = features.describe(include="all")
        summary.to_csv(f"{feature_filename.split('.')[0].split('_')[0]}_summary.csv")
    else:
        print("Loading features...")
        features = pd.read_csv(feature_filename, index_col="index")

    return features


def perform_ablation_study(
    colnames: list,
    full_train_set: pd.DataFrame,
    full_dev_set: pd.DataFrame,
    full_test_set: pd.DataFrame,
    groups: bool = False,
    reverse: bool = False,
) -> None:
    """Perform an ablation study

    Args:
        colnames (list): Names of the feature columns in the datasets
        full_train_set (pd.DataFrame): Train set
        full_dev_set (pd.DataFrame): Dev set
        full_test_set (pd.DataFrame): Test set
        groups (bool, optional): If the features should be grouped. Defaults to False.
        reverse (bool, optional): When true, the classifiers are trained for each individual feature(group),
            insead of leaving out that feature(group). Defaults to False.
    """
    train_results = {}
    dev_results = {}
    test_results = {}
    train_y = full_train_set["author"].to_numpy()
    dev_y = full_dev_set["author"].to_numpy()
    test_y = full_test_set["author"].to_numpy()
    features = ["d", "c", "p", "g", "i", "o"] if groups else colnames
    for feature in features:
        if groups:
            if reverse:
                ablation_colnames = [
                    colname for colname in colnames if colname[2] == feature
                ]
            else:
                ablation_colnames = [
                    colname for colname in colnames if colname[2] != feature
                ]
        else:
            if reverse:
                ablation_colnames = [feature]
            else:
                ablation_colnames = deepcopy(colnames)
                ablation_colnames.remove(feature)

        train_x = full_train_set[ablation_colnames].to_numpy()
        dev_x = full_dev_set[ablation_colnames].to_numpy()
        test_x = full_test_set[ablation_colnames].to_numpy()

        classifier = Classifier()
        classifier.fit(train_x=train_x, train_y=train_y)

        train_results[feature] = classifier.get_f1_score(eval_x=train_x, eval_y=train_y)
        dev_results[feature] = classifier.get_f1_score(eval_x=dev_x, eval_y=dev_y)
        test_results[feature] = classifier.get_f1_score(eval_x=test_x, eval_y=test_y)

    settings_str = f"{'_groups' if groups else ''}{'_reverse' if reverse else ''}"
    results = {
        "train_results": train_results,
        "dev_results": dev_results,
        "test_results": test_results,
    }
    with open(f"results{settings_str}.json", "w") as outfile:
        json.dump(results, outfile, indent=4)

    plot_ablation_results(train_results, name="Train", settings_str=settings_str)
    plot_ablation_results(dev_results, name="Dev", settings_str=settings_str)
    plot_ablation_results(test_results, name="Test", settings_str=settings_str)


def plot_ablation_results(
    ablation_results: dict, name: str = "", settings_str: str = ""
) -> None:
    """Plot the results of an ablation study

    Args:
        ablation_results (dict): Results of the ablation study. Keys are feature(group) names. Values are F1 scores
        name (str, optional): Name of the dataset the classifiers were evaluated on. Defaults to "".
        settings_str (str, optional): Settings of the ablation study. Defaults to "".
    """

    if "groups" in settings_str:
        x_ticks = [
            "Default Counts",
            "Complexity",
            "POS tags",
            "Grammar/spelling",
            "Punctuation",
            "Other",
        ]
        rotation = 45
        bottom = 0.3
        figsize = (7, 5)
    else:
        x_ticks = [i[4:] for i in ablation_results.keys()]
        rotation = 90
        bottom = 0.45
        figsize = (10, 5)

    plt.figure(figsize=figsize)
    plt.bar(x_ticks, ablation_results.values())
    plt.xticks(rotation=rotation)
    plt.xlabel("Missing feature")
    plt.ylabel("F1 score")
    plt.title(
        f"{name} - Ablation study for authorship attribution ({' '.join(settings_str.split('_')).strip()})"
    )
    plt.subplots_adjust(top=0.9, bottom=bottom)
    plt.savefig(f"ablation_plot_{name.lower()}{settings_str}.png")
    plt.close()


main()

## **4. Preprocessing**

### **4.1. Lowercase all letters**

Lowercase all letters in your chosen data. In that way, you remove the variety of capitalization and make the search case-insensitive, e.g. 'Cat' and 'cat' will be treated the same.

In [ ]:
# Lowercase letters in token data
updated_data = []
for book in data:
    updated_book = [w.lower() for w in book]
    updated_data.append(updated_book)
data = updated_data

# Lowercase letters in sentence data
updated_data_sents = []
for book in data_sents:
    book_sents = []
    for sent in book:
        updated_sent = [w.lower() for w in sent]
        book_sents.append(updated_sent)
    updated_data_sents.append(book_sents)
data_sents = updated_data_sents

print("Example of how the data lowercased looks like:")
print(data[0][0:10])

### **4.2. Remove stopwords**

Stop words are a set of high frequent words, often function words, in a language (examples in English: “a”, “the”, “is”, “me”). Since they occur many times in most texts, they carry very little semantic information, and are therefore often removed.

N.B. On the contrary, in the study of literary style, stylometry, stop words are usually very useful!

In [ ]:
stop_words = stopwords.words("english")

# Remove stopwords in token data
updated_data = []
for book in data:
    updated_book = [w for w in book if w.lower() not in stop_words]
    updated_data.append(updated_book)
data = updated_data

# Remove stopwords in sentence data
updated_data_sents = []
for book in data_sents:
    book_sents = []
    for sent in book:
        updated_sent = [w.lower() for w in sent if w.lower() not in stop_words]
        book_sents.append(updated_sent)
    updated_data_sents.append(book_sents)
data_sents = updated_data_sents

print("Example of how the data without stopwords looks like:")
print(data[0][0:10])

### **4.3. Remove punctuation**

Punctuation are often removed for the same reason as stopwords: to reduce tokens that hold less information.

In [ ]:
puncts = list(string.punctuation)

# Remove punctuation in token data
updated_data = []
for book in data:
    updated_book = [w for w in book if w not in puncts]
    updated_data.append(updated_book)
data = updated_data

# Remove punctuation in sentence data
updated_data_sents = []
for book in data_sents:
    book_sents = []
    for sent in book:
        updated_sent = [w.lower() for w in sent if w not in puncts]
        book_sents.append(updated_sent)
    updated_data_sents.append(book_sents)
data_sents = updated_data_sents

print("Example of how the data without punctuation looks like:")
print(data[0][0:10])

### **4.4. Annotate tokens with part-of-speech tags (needed for lemmatization)**

Part-of-speech (pos) tagging makes it possible to disambiguate words that have multiple meanings. It helps to identify the function of each word in a sentence or phrase, which is necessary in most NLP applications.

We use a simplified tagset from NLTK, see more about the tagset in table 2.1 here: https://www.nltk.org/book/ch05.html

In [ ]:
pos_tagging = True

updated_data = []
updated_data_sents = []

for book in data_sents:
    book_pos = []
    book_pos_sents = []
    for sent in book:
        sent_pos = []
        sent_tuples = nltk.pos_tag(
            sent, tagset="universal"
        )  # simplified universal pos tagset, see table 2.1 in https://www.nltk.org/book/ch05.html
        for st in sent_tuples:
            s = (
                st[0] + "_" + st[1]
            )  # change tuple to string, e.g. ('alice', 'NOUN') --> 'alice_NOUN'
            sent_pos.append(s)
        book_pos += sent_pos
        book_pos_sents.append(sent_pos)
    updated_data.append(book_pos)
    updated_data_sents.append(book_pos_sents)

data = updated_data
data_sents = updated_data_sents  # List of books, each book list of sentences, each sentences list of (token, pos) tuple

print("Example of how the data with pos tags looks like:")
print(data[0][0:10])

### **4.5. Lemmatize tokens**

Lemmatization is the process of converting words to their base forms (dictionary forms). This is used so that inflected forms of a word can be analysed as a single item, e.g. 'cat' and 'cats' have the same lemma 'cat'.

To choose the correct lemma, we need information about the word's meaning, given that words can have multiple meanings, e.g.:
- leaves_VERB --> leave_VERB
- leaves_NOUN --> leaf_NOUN

For this reason, we include part-of-speech information.



In [ ]:
wnl = nltk.WordNetLemmatizer()

if pos_tagging:
    updated_data = []
    updated_data_sents = []

    for book in data_sents:  # loop through each book in data_pos
        book_lemmas = []
        book_lemmas_sents = []
        for sent in book:  # loop through each sentence in book
            sent_update = []
            for token_pos in sent:  # loop through each (token, pos) tuple in sentence
                split_token = token_pos.rsplit("_", 1)
                if "NOUN" in token_pos:
                    token_pos = (
                        wnl.lemmatize(split_token[0].lower(), pos="n")
                        + "_"
                        + split_token[1]
                    )
                if "VERB" in token_pos:
                    token_pos = (
                        wnl.lemmatize(split_token[0].lower(), pos="v")
                        + "_"
                        + split_token[1]
                    )
                if "ADJ" in token_pos:
                    token_pos = (
                        wnl.lemmatize(split_token[0].lower(), pos="a")
                        + "_"
                        + split_token[1]
                    )
                if "ADV" in token_pos:
                    token_pos = (
                        wnl.lemmatize(split_token[0].lower(), pos="r")
                        + "_"
                        + split_token[1]
                    )
                sent_update.append(token_pos)
                book_lemmas.append(token_pos)
            book_lemmas_sents.append(sent_update)
        updated_data.append(book_lemmas)
        updated_data_sents.append(book_lemmas_sents)

    data = updated_data
    data_sents = updated_data_sents

print("Example of how the lemmatized data looks like:")
print(data[0][0:10])

## **5. Do more statistics of your choice**

In sections 2.1 and 2.2, you obtained basic statistics for any selected dataset. Perhaps those metrics were sufficient for your chosen study. If not (which is likely) , additional blocks of code are provided below to enable you to conduct further statistical analyses on your selected data.

### **5.1. Get the most common terms in your data according to their relative frequencies**

One simple method to assess the importance of a term in a book is by looking at its relative frequency. Here, we compute the relative frequencies for all terms in a book and print the top terms with the highest relative frequency.

Before you run the code, you have to decide:
1. how many top terms you want to print
2. if you want to use part-of-speech-tags or not


Make changes in the code below accordingly. Make sure that you describe and motivate your choices in your lab report.

In [ ]:
no_top_terms = 10  # <------------------------------------------------  1. CHOOSE THE NUMBER OF TOP RELATIVE FREQUENT WORDS!

# Choose if you want to use pos tags       <------------------------------- 2. CHANGE False --> True IF YOU WANT TO USE POS TAGS!
use_pos = False

if use_pos:
    if not pos_tagging:
        print(
            "You have to perform part-of-speech tagging (4.4.) if you want to includ pos tags!"
        )

# Calculate the total number of terms in the text
all_terms = []
for book in data:
    for t in book:
        if pos_tagging and not use_pos:
            t = t.rsplit("_", 1)[0]
        all_terms.append(t)
total_terms = len(all_terms)

# Calculate the frequency of each word
term_counts = Counter(all_terms)

# Calculate the relative frequency of each term
relative_frequencies_terms = dict()
for term, count in term_counts.items():
    relative_frequencies_terms[term] = count / total_terms

# Sort the terms based on relative frequency in descending order
sorted_terms = sorted(
    relative_frequencies_terms.items(), key=lambda x: x[1], reverse=True
)

# Get the top N terms
top_terms = sorted_terms[:no_top_terms]

# Print results
print(
    "Top",
    no_top_terms,
    "terms and their relative frequencies:",
)
print("")
for term, frequency in top_terms:
    print(term, round(frequency, 10))


### **5.2. Calculate the relative frequencies of specified terms**

In this section, we also examine relative frequencies (as in 5.1.). However, instead of focusing on the top terms, we compute the relative frequencies for any terms of interest. This enables us to compare the use of specific terms in our chosen book(s).

Before you run the code, you have to decide:
1. for wich terms you want to calculate the relative frequencies
2. if you want to use part-of-speech-tags or not

Make changes in the code below accordingly. Make sure that you describe and motivate your choices in your lab report.

In [ ]:
terms = [
    "summer",
    "autumn",
    "winter",
    "spring",
]  # <---------------------------- 1. CHOOSE TERMS! ADD POS TAGS IF YOU WANT TO INCLUDE THAT, E.G. 'time_NOUN'

# Choose if you want to use pos tags       <----------------------------------------- 2. CHANGE False --> True IF YOU WANT TO USE POS TAGS!
use_pos = False

if use_pos:
    if not pos_tagging:
        print(
            "You have to perform part-of-speech tagging (4.4.) if you want to includ pos tags!"
        )

# Calculate the total number of terms in the text
all_terms = []
for book in data:
    for t in book:
        if pos_tagging and not use_pos:
            t = t.rsplit("_", 1)[0]
        all_terms.append(t)
total_terms = len(all_terms)

# Calculate the frequency of each term
term_counts = Counter(all_terms)

# Calculate the relative frequency of each term
relative_frequencies_terms = dict()
for term, count in term_counts.items():
    relative_frequencies_terms[term] = count / total_terms

# rel_freqs = {term: count / total_terms for term, count in term_counts.items()}
# relative_frequencies_pos = dict()

# Print results
print("The relative frequencies for selected terms:")
for term in terms:
    print(term, ":", relative_frequencies_terms[term])


### **5.3. Plot relative frequencies of term(s) in a book**

In this section, we use the relative frequencies of term(s) or character(s) to study their occurrences over time in a book. We cut the book into equal 'chunks' so that we can plot the the mentions of the terms or characters throughout the course of the book. If your dataset includes more than one book, the code will generate a separate graph for each book.

Before you run the code, you have to decide:
1. if you want to use part-of-speech-tags or not
2. how many chunks you want to cut the book(s) into
3. what term(s) or character(s) you want to plot

Make changes in the code below accordingly. Make sure that you describe and motivate your choices in your lab report.

In [ ]:
# Choose if you want to use pos tags       <-----------------------------------------------  1. CHANGE False --> True IF YOU WANT TO USE POS TAGS!
use_pos = False

if use_pos:
    if not pos_tagging:
        print(
            "You have to perform part-of-speech tagging (4.4.) if you want to includ pos tags!"
        )


def plot_relative_frequencies(book_tokens, title, use_pos=False):
    no_chunks = 20  # Change to whatever number of chunks you want     <---------------------- 2. CHOOSE NUMBER OF CHUNKS!

    # Calculate the approximate number of tokens per chunk
    tokens_per_chunk = len(book_tokens) // no_chunks

    # If pos tags exist in the data, but is chosen not to be used (use_pos = False)
    if not use_pos:
        if pos_tagging:
            if "_" in book_tokens[0]:
                books_tokens_no_pos = []
                for t in book_tokens:
                    books_tokens_no_pos.append(t.rsplit("_", 1)[0])
        book_tokens = books_tokens_no_pos

    # Divide the text into chunks
    chunks = []
    start_index = 0
    for i in range(no_chunks - 1):
        end_index = start_index + tokens_per_chunk
        chunks.append(" ".join(book_tokens[start_index:end_index]))
        start_index = end_index

    # Add the remaining words to the last chunk
    chunks.append(" ".join(book_tokens[start_index:]))

    # Choose which term(s) that you want to study    <---------------------------------------- 3. CHOOSE TERMS! ADD POS TAGS IF YOU WANT TO INCLUDE THAT, E.G. 'time_NOUN'
    words_to_plot = [["time"], ["rabbit"], ["heart"]]

    # Check that the use of pos tags in words to plot and in the data is consistent
    if use_pos:
        for word in words_to_plot[0]:
            if "_" not in word:
                print("You have chosen to use pos tags (use_pos = True).")
                print(
                    "You have to include pos tags in your words to plot, or change use_pos = False!"
                )
                return None
    else:
        for word in words_to_plot[0]:
            if "_" in word:
                print("You have chosen to not use pos tags (use_pos = False).")
                print(
                    "You have to exclude pos tags in your words to plot, or change use_pos = True!"
                )
                return None

    # Calculate the relative frequencies of your words in each text chunk
    all_relative_frequencies = OrderedDict()
    index = 0
    for chunk in chunks:
        index += 1
        word_counts = []
        chunk_tokens = word_tokenize(chunk)

        # The total number of words in the chunk
        no_tokens_in_chunk = len(chunk_tokens)

        # Calculate the relative frequencies
        relative_frequencies = []
        for words in words_to_plot:
            no_apperences = 0
            for token in chunk_tokens:
                if token in words or token.lower() in words:
                    no_apperences += 1
            if no_apperences > 0:
                relative_frequency = no_apperences / no_tokens_in_chunk
            else:
                relative_frequency = 0
            relative_frequencies.append(relative_frequency)
        all_relative_frequencies[str(index)] = relative_frequencies

    # Plot the relative frequencies of the words throughout the chunks of the book
    col = ["dodgerblue", "red", "black", "orange", "darkgreen", "darkgray", "maroon"]
    mrkr = ["+", "v", "o", "v", "o", "^", "s", ">", "o"]
    plt.figure(figsize=(10, 6))
    for i, word in enumerate(words_to_plot):
        freqs = []
        for chunk in all_relative_frequencies:
            freqs.append(all_relative_frequencies[chunk][i])
        plt.plot(
            range(1, no_chunks + 1), freqs, label=word, color=col[i], marker=mrkr[i]
        )

    plt.xlabel("Book Chunk Number")
    plt.ylabel("Relative Frequency")
    plt.title(title)
    plt.xticks(range(1, no_chunks + 1))
    plt.legend()
    plt.tight_layout()
    plt.show()


for i in range(0, len(books)):
    plot_relative_frequencies(data[i], books[i], use_pos)

### **5.4. Plot the part-of-speech distribution**

As a final option, this code allows you to plot the part-of-speech distribution of your chosen data. This could be useful if you, for example, are interested in doing an analysis of linguistic style (stylometry) of an author, a genre or a book.

It is necessary that you perform part-of-speech tagging before you run the code below. And, as allready mentioned, describe and motivate your choice of statistic analysis in your lab report.

In [ ]:
if not pos_tagging:
    print(
        "You have to perform part-of-speech tagging (4.4.) if you want to includ pos tags!"
    )

# Calculate the total number of terms in the text
all_pos = []
for book in data:
    for t in book:
        pos = t.rsplit("_", 1)[1]
        all_pos.append(pos)
total_pos = len(all_pos)

# Calculate the frequency of each pos tag
pos_counts = Counter(all_pos)

# Calculate the relative frequency of each term
relative_frequencies_pos = dict()
for term, count in pos_counts.items():
    relative_frequencies_pos[term] = count / total_pos

# Plot the bar chart
plt.bar(relative_frequencies_pos.keys(), relative_frequencies_pos.values())
plt.xlabel("Part-of-Speech")
plt.ylabel("Relative Frequency")
plt.title("Part-of-Speech Distribution")
for pos, frequency in relative_frequencies_pos.items():
    plt.text(pos, frequency + 0.0001, f"{frequency:.3f}", ha="center", va="bottom")
plt.show()